# Chapter 4 — Stacks and Queues

*Built with [Claude](https://claude.ai) by Anthropic.*

Stacks (LIFO) and queues (FIFO) show up constantly in parsing, scheduling, and stream processing — all territory familiar to data scientists.

In [12]:
from collections import deque

def show_stack(stack, label='stack'):
    """Print a stack with top on the right."""
    inner = ', '.join(repr(x) for x in stack)
    print(f"  {label:10s}: [{inner}]  ← top")

def show_mono(stack, label='mono'):
    """Print monotonic stack contents."""
    inner = ', '.join(str(x) for x in stack)
    print(f"  {label:10s}: [{inner}]")

---

## Part 1 — Stack for Matching / Validity

**DS/ML connection:** XML/JSON parsing, validating nested configuration files, checking balanced parentheses in generated code — all use a stack to track open context that must be closed in LIFO order.

In [13]:
# ── DS/ML PARALLEL: parsing nested structures ──
import json

def is_valid_json(s):
    try:
        json.loads(s)
        return True
    except json.JSONDecodeError:
        return False

samples = ['{"a": [1, 2]}', '{"a": [1, 2}', '[]']
for s in samples:
    print(f"  {s!r:25s}  valid={is_valid_json(s)}")

  '{"a": [1, 2]}'            valid=True
  '{"a": [1, 2}'             valid=False
  '[]'                       valid=True


### LC 20 — [Valid Parentheses](https://leetcode.com/problems/valid-parentheses/)

**Problem:** Given a string of `()[]{}`, return `True` if every opening bracket is closed in the correct order.

**DS parallel:** Validating nested config keys, checking generated SQL/code for balanced delimiters.

**Key insight:** Push opening brackets onto a stack. When you see a closing bracket, pop and check that it matches. If the stack is empty at the end, all brackets were matched.

In [14]:
# ── DS/ML APPROACH: regex / parser ──
import re

def is_valid_regex(s):
    # Repeatedly collapse innermost matched pairs until nothing changes
    prev = None
    while prev != s:
        prev = s
        s = re.sub(r'\(\)|\[\]|\{\}', '', s)
    return s == ''

# ── RAW PYTHON (interview answer) ──
def isValid(s: str) -> bool:
    stack = []
    match = {')': '(', ']': '[', '}': '{'}
    for ch in s:
        if ch in '([{':
            stack.append(ch)
        elif ch in ')]}':
            if not stack or stack[-1] != match[ch]:
                return False
            stack.pop()
    return len(stack) == 0

tests = ['()', '()[]{', '(]', '([)]', '{[]}']
for t in tests:
    r = isValid(t)
    rr = is_valid_regex(t)
    print(f"  {t!r:12s}  stack={r}  regex={rr}")

  '()'          stack=True  regex=True
  '()[]{'       stack=False  regex=False
  '(]'          stack=False  regex=False
  '([)]'        stack=False  regex=False
  '{[]}'        stack=True  regex=True


In [15]:
# ── TRACE: LC 20 ──
def isValid_trace(s):
    stack = []
    match = {')': '(', ']': '[', '}': '{'}
    print(f"  Input: {s!r}")
    print(f"  {'step':>4}  {'ch':>4}  action")
    print(f"  {'----':>4}  {'--':>4}  ------")
    for i, ch in enumerate(s):
        if ch in '([{':
            stack.append(ch)
            action = f"push → {stack}"
        else:
            if not stack:
                action = 'FAIL: empty stack'
                print(f"  {i:>4}  {ch:>4}  {action}")
                return False
            top = stack[-1]
            if top != match[ch]:
                action = f"FAIL: top={top!r} != expected={match[ch]!r}"
                print(f"  {i:>4}  {ch:>4}  {action}")
                return False
            stack.pop()
            action = f"pop  → {stack}"
        print(f"  {i:>4}  {ch:>4}  {action}")
    result = len(stack) == 0
    print(f"  final stack: {stack}  → {result}")
    return result

isValid_trace('({[]})')
print()
isValid_trace('([)]')

  Input: '({[]})'
  step    ch  action
  ----    --  ------
     0     (  push → ['(']
     1     {  push → ['(', '{']
     2     [  push → ['(', '{', '[']
     3     ]  pop  → ['(', '{']
     4     }  pop  → ['(']
     5     )  pop  → []
  final stack: []  → True

  Input: '([)]'
  step    ch  action
  ----    --  ------
     0     (  push → ['(']
     1     [  push → ['(', '[']
     2     )  FAIL: top='[' != expected='('


False

### LC 394 — [Decode String](https://leetcode.com/problems/decode-string/)

**Problem:** Given an encoded string like `"3[a2[c]]"`, decode it to `"accaccacc"`.

**DS parallel:** Expanding compressed feature representations, templating repeated config blocks.

**Key insight:** Use two stacks — one for counts, one for partial strings. When `[` is seen, push current state. When `]` is seen, pop and expand.

In [16]:
# ── DS/ML APPROACH: regex-based expansion ──
import re

def decode_regex(s):
    # Iteratively expand innermost brackets
    while '[' in s:
        s = re.sub(r'(\d+)\[([a-z]*)\]', lambda m: m.group(2) * int(m.group(1)), s)
    return s

# ── RAW PYTHON (interview answer) ──
def decodeString(s: str) -> str:
    count_stack = []
    str_stack = []
    curr_str = ''
    curr_num = 0
    for ch in s:
        if ch.isdigit():
            curr_num = curr_num * 10 + int(ch)
        elif ch == '[':
            count_stack.append(curr_num)
            str_stack.append(curr_str)
            curr_num = 0
            curr_str = ''
        elif ch == ']':
            times = count_stack.pop()
            prev  = str_stack.pop()
            curr_str = prev + curr_str * times
        else:
            curr_str += ch
    return curr_str

tests = ['3[a]2[bc]', '3[a2[c]]', '2[abc]3[cd]ef']
for t in tests:
    r  = decodeString(t)
    rr = decode_regex(t)
    print(f"  {t!r:20s}  stack={r!r}  regex={rr!r}")

  '3[a]2[bc]'           stack='aaabcbc'  regex='aaabcbc'
  '3[a2[c]]'            stack='accaccacc'  regex='accaccacc'
  '2[abc]3[cd]ef'       stack='abcabccdcdcdef'  regex='abcabccdcdcdef'


In [17]:
# ── TRACE: LC 394 ──
def decodeString_trace(s):
    count_stack = []
    str_stack = []
    curr_str = ''
    curr_num = 0
    print(f"  Input: {s!r}")
    print(f"  {'ch':>4}  {'curr_num':>8}  {'curr_str':>12}  count_stk  str_stk")
    for ch in s:
        if ch.isdigit():
            curr_num = curr_num * 10 + int(ch)
        elif ch == '[':
            count_stack.append(curr_num)
            str_stack.append(curr_str)
            curr_num = 0
            curr_str = ''
        elif ch == ']':
            times = count_stack.pop()
            prev  = str_stack.pop()
            curr_str = prev + curr_str * times
        else:
            curr_str += ch
        print(f"  {ch:>4}  {curr_num:>8}  {curr_str!r:>12}  {count_stack}  {str_stack}")
    print(f"  Result: {curr_str!r}")

decodeString_trace('3[a2[c]]')

  Input: '3[a2[c]]'
    ch  curr_num      curr_str  count_stk  str_stk
     3         3            ''  []  []
     [         0            ''  [3]  ['']
     a         0           'a'  [3]  ['']
     2         2           'a'  [3]  ['']
     [         0            ''  [3, 2]  ['', 'a']
     c         0           'c'  [3, 2]  ['', 'a']
     ]         0         'acc'  [3]  ['']
     ]         0   'accaccacc'  []  []
  Result: 'accaccacc'


---

## Part 2 — Monotonic Stack

**DS/ML connection:** Computing rolling min/max efficiently, finding the nearest larger/smaller value in a time series — these are monotonic stack operations. The pattern underlies many streaming feature computations.

In [18]:
# ── DS/ML PARALLEL: pandas rolling operations ──
import pandas as pd
import numpy as np

temps = [73, 74, 75, 71, 69, 72, 76, 73]
s = pd.Series(temps)

# For each day, how many days until a warmer day?
# Pandas doesn't have a direct 'next larger' operation,
# but rolling max gives you the window perspective:
print('  temperatures:', temps)
print('  rolling max (3-day):', s.rolling(3, min_periods=1).max().tolist())

  temperatures: [73, 74, 75, 71, 69, 72, 76, 73]
  rolling max (3-day): [73.0, 74.0, 75.0, 75.0, 75.0, 72.0, 76.0, 76.0]


### LC 739 — [Daily Temperatures](https://leetcode.com/problems/daily-temperatures/)

**Problem:** Given daily temperatures, return an array where `answer[i]` is the number of days until a warmer temperature. If no warmer day exists, put `0`.

**DS parallel:** "How many rows until the next spike?" in a time series — next-greater-element queries on streaming data.

**Key insight:** Use a monotonic decreasing stack of indices. When a warmer temperature is found, pop all cooler indices and fill their answers.

In [19]:
# ── DS/ML APPROACH: vectorized next-greater via numpy ──
def daily_temps_numpy(temperatures):
    arr = np.array(temperatures)
    n = len(arr)
    result = np.zeros(n, dtype=int)
    # O(n^2) brute force — straightforward but slow
    for i in range(n):
        for j in range(i + 1, n):
            if arr[j] > arr[i]:
                result[i] = j - i
                break
    return result.tolist()

# ── RAW PYTHON (interview answer) ──
def dailyTemperatures(temperatures):
    n = len(temperatures)
    answer = [0] * n
    stack = []   # indices, decreasing temperatures
    for i, temp in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < temp:
            j = stack.pop()
            answer[j] = i - j
        stack.append(i)
    return answer

temps = [73, 74, 75, 71, 69, 72, 76, 73]
r1 = daily_temps_numpy(temps)
r2 = dailyTemperatures(temps)
print('  temperatures:', temps)
print('  numpy O(n²): ', r1)
print('  mono stack:  ', r2)

  temperatures: [73, 74, 75, 71, 69, 72, 76, 73]
  numpy O(n²):  [1, 1, 4, 2, 1, 1, 0, 0]
  mono stack:   [1, 1, 4, 2, 1, 1, 0, 0]


In [20]:
# ── TRACE: LC 739 ──
def dailyTemperatures_trace(temperatures):
    n = len(temperatures)
    answer = [0] * n
    stack = []
    print(f"  temps: {temperatures}")
    print(f"  {'i':>3}  {'temp':>5}  {'action':<40}  stack")
    for i, temp in enumerate(temperatures):
        actions = []
        while stack and temperatures[stack[-1]] < temp:
            j = stack.pop()
            answer[j] = i - j
            actions.append(f"pop {j}→ans[{j}]={i-j}")
        stack.append(i)
        actions.append(f"push {i}")
        print(f"  {i:>3}  {temp:>5}  {', '.join(actions):<40}  {stack}")
    print(f"  answer: {answer}")

dailyTemperatures_trace([73, 74, 75, 71, 69, 72, 76, 73])

  temps: [73, 74, 75, 71, 69, 72, 76, 73]
    i   temp  action                                    stack
    0     73  push 0                                    [0]
    1     74  pop 0→ans[0]=1, push 1                    [1]
    2     75  pop 1→ans[1]=1, push 2                    [2]
    3     71  push 3                                    [2, 3]
    4     69  push 4                                    [2, 3, 4]
    5     72  pop 4→ans[4]=1, pop 3→ans[3]=2, push 5    [2, 5]
    6     76  pop 5→ans[5]=1, pop 2→ans[2]=4, push 6    [6]
    7     73  push 7                                    [6, 7]
  answer: [1, 1, 4, 2, 1, 1, 0, 0]


### LC 496 — [Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/)

**Problem:** Given two arrays `nums1` (subset of `nums2`), find the next greater element for each value in `nums1` within `nums2`. Return `-1` if none exists.

**DS parallel:** Lookup-based enrichment — for each row in a left table, find the next event in a right table.

**Key insight:** Build a `next_greater` map for all elements in `nums2` using a monotonic stack, then answer all `nums1` queries in O(1) each.

In [21]:
# ── DS/ML APPROACH: pandas merge + shift ──
import pandas as pd

def next_greater_pandas(nums1, nums2):
    # Brute-force: for each value in nums1, scan nums2
    result = []
    for val in nums1:
        idx = nums2.index(val)
        found = -1
        for j in range(idx + 1, len(nums2)):
            if nums2[j] > val:
                found = nums2[j]
                break
        result.append(found)
    return result

# ── RAW PYTHON (interview answer) ──
def nextGreaterElement(nums1, nums2):
    # Pre-compute next_greater for all elements in nums2
    next_greater = {}
    stack = []
    for num in nums2:
        while stack and stack[-1] < num:
            next_greater[stack.pop()] = num
        stack.append(num)
    # Remaining in stack have no next greater
    while stack:
        next_greater[stack.pop()] = -1
    return [next_greater[n] for n in nums1]

nums1 = [4, 1, 2]
nums2 = [1, 3, 4, 2]
r1 = next_greater_pandas(nums1, nums2)
r2 = nextGreaterElement(nums1, nums2)
print(f"  nums1={nums1}, nums2={nums2}")
print(f"  brute force: {r1}")
print(f"  mono stack:  {r2}")

  nums1=[4, 1, 2], nums2=[1, 3, 4, 2]
  brute force: [-1, 3, -1]
  mono stack:  [-1, 3, -1]


In [22]:
# ── TRACE: LC 496 ──
def nextGreaterElement_trace(nums1, nums2):
    next_greater = {}
    stack = []
    print(f"  nums2: {nums2}")
    print(f"  Building next_greater map:")
    print(f"  {'num':>5}  {'action':<35}  stack")
    for num in nums2:
        actions = []
        while stack and stack[-1] < num:
            popped = stack.pop()
            next_greater[popped] = num
            actions.append(f"next_greater[{popped}]={num}")
        stack.append(num)
        actions.append(f"push {num}")
        print(f"  {num:>5}  {', '.join(actions):<35}  {stack}")
    while stack:
        next_greater[stack.pop()] = -1
    print(f"  next_greater map: {next_greater}")
    print(f"  nums1={nums1} → {[next_greater[n] for n in nums1]}")

nextGreaterElement_trace([4, 1, 2], [1, 3, 4, 2])

  nums2: [1, 3, 4, 2]
  Building next_greater map:
    num  action                               stack
      1  push 1                               [1]
      3  next_greater[1]=3, push 3            [3]
      4  next_greater[3]=4, push 4            [4]
      2  push 2                               [4, 2]
  next_greater map: {1: 3, 3: 4, 2: -1, 4: -1}
  nums1=[4, 1, 2] → [-1, 3, -1]
